In [1]:
pip install sentence-transformers transformers torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 1.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 2.4 MB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 5.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 3.9 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/4 [huggingface-hub]
    Found existing installation: tokenizers 0.21.40/4 [huggingface-hub]
    Uninstalling tokenizers-0.21.4:━━━━━━━━━ 0/4 [huggingface-hub]
      Successfully uninstalled tokenizers-0.21.4 0/4 [huggingface-hub]
  Attempting uninstall: transformers━━━━━━━━ 0/4 [huggingface-hub]
    Found existing installation: transformers 4.53.02m0/4 [huggingface-hub]
    Uninstalling transformers-4.53.0:╺━━━━━━━━━

In [1]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from sklearn.metrics.pairwise import cosine_similarity
import torch


# STEP 1: CREATE DOCUMENT COLLECTION

documents = {
    "OSI and TCP/IP Models": [
        """
        The OSI model is a conceptual framework used to understand how
        communication takes place between different systems in a computer
        network. It consists of seven layers: Physical, Data Link, Network,
        Transport, Session, Presentation, and Application.
        """,

        """
        The TCP/IP model is a practical networking model used for communication
        over the Internet. It consists of Network Access, Internet, Transport,
        and Application layers. Protocols such as IP, TCP, UDP, HTTP, and DNS
        operate within different layers of the TCP/IP model.
        """
    ],

    "Physical and Data Link Layer": [
        """
        The Physical Layer is the lowest layer of the OSI model and is
        responsible for transmitting raw bits over a physical medium. It deals
        with transmission media, signals, connectors, data rates, and the
        physical transmission of data between network devices.
        """,

        """
        The Data Link Layer provides reliable communication between directly
        connected devices. It organizes data into frames and uses MAC
        addresses to identify devices. It also performs error detection and
        controls access to the shared communication medium.
        """
    ],

    "Network Layer and Routing": [
        """
        The Network Layer is responsible for delivering packets from a source
        network to a destination network. It uses logical addressing such as
        IP addresses to identify source and destination devices. IPv4 and IPv6
        are important protocols associated with this layer.
        """,

        """
        Routing is the process of determining a suitable path for packets to
        travel from the source to the destination. Routers use routing tables
        and routing algorithms to make forwarding decisions. Routing protocols
        such as RIP, OSPF, and BGP help exchange information about network paths.
        """
    ],

    "Transport Layer TCP and UDP": [
        """
        The Transport Layer provides communication between applications
        running on different hosts. TCP and UDP are two important Transport
        Layer protocols. TCP is connection-oriented and provides reliable,
        ordered delivery of data between communicating systems.
        """,

        """
        TCP uses acknowledgements, retransmission, and flow control to provide
        reliable communication. UDP is a connectionless transport protocol
        with lower overhead and does not provide the same reliability
        mechanisms as TCP. The choice between TCP and UDP depends on the
        requirements of the application.
        """
    ],

    "Application Layer HTTP DNS and FTP": [
        """
        The Application Layer provides network services directly to user
        applications. HTTP is used for communication between web clients and
        web servers. HTTPS provides secure communication for web applications
        by using HTTP with encryption.
        """,

        """
        DNS stands for Domain Name System and is used to translate domain names
        into IP addresses. FTP is a protocol used for transferring files
        between computers over a network. HTTP, DNS, and FTP provide important
        services to network applications.
        """
    ]
}


# STEP 2: COMBINE PARAGRAPHS
document_names = list(documents.keys())

document_texts = []

for name in document_names:
    text = " ".join(documents[name])
    document_texts.append(text)


# STEP 3: LOAD SENTENCE EMBEDDING MODEL
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# STEP 4: CREATE TRUE EMBEDDINGS
document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True
)


# STEP 5: LOAD EXTRACTIVE QA MODEL
print("Loading Question Answering model...")

qa_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)


# STEP 6: TAKE QUESTION FROM USER
question = input("\nEnter your Computer Networks question: ")


# STEP 7: CREATE QUESTION EMBEDDING
question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
)


# STEP 8: CALCULATE COSINE SIMILARITY
similarities = cosine_similarity(
    question_embedding,
    document_embeddings
)[0]


# STEP 9: FIND MOST RELEVANT DOCUMENT
best_index = similarities.argmax()

best_document = document_names[best_index]

best_score = similarities[best_index]

best_context = document_texts[best_index]


# STEP 10: DISPLAY SEMANTIC SEARCH RESULT
print("\nSEMANTIC SEARCH RESULT\n")

print("Question:", question)

print("Most Relevant Document:", best_document)

print("Cosine Similarity:", round(float(best_score), 4))


# STEP 11: DISPLAY ALL SIMILARITY SCORES
print("\nDocument Similarity Scores:")

for i in range(len(document_names)):
    print(
        document_names[i],
        "->",
        round(float(similarities[i]), 4)
    )


# STEP 12: TOKENIZE QUESTION AND CONTEXT
inputs = qa_tokenizer(
    question,
    best_context,
    return_tensors="pt",
    truncation=True
)


# STEP 13: RUN EXTRACTIVE QA MODEL
with torch.no_grad():
    outputs = qa_model(**inputs)


# STEP 14: FIND ANSWER START AND END POSITIONS
start_scores = outputs.start_logits
end_scores = outputs.end_logits

start_index = torch.argmax(start_scores, dim=1).item()
end_index = torch.argmax(end_scores, dim=1).item()


# Make sure the end position is not before the start position
if end_index < start_index:
    end_index = start_index


# STEP 15: EXTRACT ANSWER
answer_tokens = inputs["input_ids"][0][
    start_index:end_index + 1
]

answer = qa_tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)


# STEP 16: CALCULATE CONFIDENCE
start_probability = torch.softmax(
    start_scores, dim=1
)[0][start_index]

end_probability = torch.softmax(
    end_scores, dim=1
)[0][end_index]

confidence = start_probability * end_probability


# STEP 17: DISPLAY EXTRACTIVE QA RESULT
print("\nEXTRACTIVE QUESTION ANSWERING\n")

print("Answer:", answer)

print("Confidence:", round(float(confidence), 4))

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading Question Answering model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Enter your Computer Networks question:  what is TCP?



SEMANTIC SEARCH RESULT

Question: what is TCP?
Most Relevant Document: Transport Layer TCP and UDP
Cosine Similarity: 0.6714

Document Similarity Scores:
OSI and TCP/IP Models -> 0.4856
Physical and Data Link Layer -> 0.3036
Network Layer and Routing -> 0.3336
Transport Layer TCP and UDP -> 0.6714
Application Layer HTTP DNS and FTP -> 0.456

EXTRACTIVE QUESTION ANSWERING

Answer: connection - oriented
Confidence: 0.9876
